<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/01-model-apis/03-streaming.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 流式响应

**目标：** 流式接收响应，并理解 UI 需要从流式后端拿到什么。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 准备工作

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的 setup 辅助函数，并拉入 `groq` 客户端——这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的**钥匙图标** → **Add new secret** 添加，名称严格写成 `GROQ_API_KEY`，粘贴其值，并打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整讲解与选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型——见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 为什么要流式

一个 LLM 产品里有两个延迟很关键，而且它们差别巨大：

- **首个 token 时间（TTFT）：** 用户要多久才能看到*任何东西*。在 Groq 上（它使用定制的推理硬件）通常远低于一秒。
- **总延迟：** 响应完整返回要多久。对于长回答，是好几秒。

> **💡 为什么重要 ——** 非流式 UI 让用户盯着转圈图标等完整结果；流式 UI 在 TTFT 之后就觉得响应很快。同样的模型、同样的成本，产品体验却完全不同。


In [ ]:
import time

start = time.monotonic()
first_token_at = None

stream = client.chat.completions.create(
    model=MODEL,
    max_tokens=400,
    stream=True,
    messages=[{'role': 'user', 'content': 'Explain, in ~150 words, why database indexes speed up reads but slow down writes.'}],
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        if first_token_at is None:
            first_token_at = time.monotonic() - start
        print(delta, end='', flush=True)

total = time.monotonic() - start
print(f'\n\nTTFT: {first_token_at:.2f}s   total: {total:.2f}s')


跑一下，注意两个数字之间的差距：那个差距就是流式为你买到的用户体验。每个 `chunk` 是一个 `ChatCompletionChunk`；文本增量在 `chunk.choices[0].delta.content`（在不携带文本的 chunk 上它是 `None`，比如第一个和最后一个）。

## chunk 流的解剖

Groq 遵循 OpenAI 的流式格式：一串 `ChatCompletionChunk` 对象，每个都携带一个 `delta`（相对上一个 chunk 的增量）：

| 字段 | 何时出现 | 里面是什么 |
|---|---|---|
| `delta.role` | 第一个 chunk | 通常是 `"assistant"`，content 为空 |
| `delta.content` | 很多 | 文本增量（一个或几个 token） |
| `delta.tool_calls` | 调用工具时 | tool call 的增量片段 |
| `finish_reason` | 最后一个 content chunk | `"stop"`、`"length"` 或 `"tool_calls"` |
| `x_groq.usage` | 最后一个 chunk | token 记账（Groq 扩展） |

与某些厂商不同，Groq 会在最后一个 chunk 上以 `x_groq.usage` 返回最终的 token 用量：你在同一次请求里既拿到增量展示，*也*拿到计费记录，而且是自动的（不需要额外的请求标志）。


In [ ]:
# 同一个请求，这次检查 chunk 结构。我们累积文本并读取最终用量。
text_parts = []
final_usage = None
finish_reason = None

stream = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    stream=True,
    messages=[{'role': 'user', 'content': 'Two sentences: what is connection pooling?'}],
)
for chunk in stream:
    # usage chunk 的 choices 列表是空的，所以索引前要先判断。
    if chunk.choices:
        delta = chunk.choices[0].delta
        if delta.content:
            text_parts.append(delta.content)
        if chunk.choices[0].finish_reason:
            finish_reason = chunk.choices[0].finish_reason
    # Groq 把 usage 挂在最后一个 chunk 的 x_groq 下（自动发送）。
    if getattr(chunk, 'x_groq', None) and chunk.x_groq.usage:
        final_usage = chunk.x_groq.usage

full_text = ''.join(text_parts)
print(full_text)
print()
print('finish_reason:', finish_reason)
print('final usage:', final_usage)


跑一下，注意两件事。第一，usage chunk 在内容结束*之后*才到，并携带一个空的 `choices` 列表，这就是我们在索引前要先判断 `if chunk.choices:` 的原因。第二，完整文本是你自己把 delta 拼起来的；OpenAI 的流式格式里没有一个单独的「最终消息」对象，所以边收边累积就是标准做法。

## 流式 + 工具调用

tool call 也是流式的，而这里开始变得琐碎。tool call 以*片段*的形式跨很多 chunk 到达：`id` 和函数 `name` 落在第一个片段上，随后 `arguments` 字符串在后续 chunk 里一点一点滴出来。你按 `index` 累积，等流结束后再解析。UI 就是这样在模型还在写调用时显示「Searching for: berlin weath...」的。


In [ ]:
import json

WEATHER_TOOL = {
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get current weather for a city. Call for any weather question.',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
        },
    },
}

# 按 index 累积 tool-call 片段。每个 index 对应一个 tool call。
tool_calls = {}  # index -> {'id': ..., 'name': ..., 'args': ''}

stream = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    stream=True,
    tools=[WEATHER_TOOL],
    messages=[{'role': 'user', 'content': 'What is the weather in Berlin?'}],
)
for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if not delta.tool_calls:
        continue
    for tc in delta.tool_calls:
        slot = tool_calls.setdefault(tc.index, {'id': None, 'name': None, 'args': ''})
        if tc.id:
            slot['id'] = tc.id
        if tc.function and tc.function.name:
            slot['name'] = tc.function.name
            print(f'tool call started: {tc.function.name}')
        if tc.function and tc.function.arguments:
            slot['args'] += tc.function.arguments
            print(f'  partial args so far: {slot["args"]!r}')

# 只在流结束后解析一次——流到一半时片段不是合法 JSON。
for idx, slot in tool_calls.items():
    args = json.loads(slot['args'] or '{}')
    print(f'tool call complete: {slot["name"]}({args})  id={slot["id"]}')


跑一下，注意那些 `partial args` 片段：在流结束之前它们*不是*合法 JSON，所以绝不要在流中途 `json.loads`。实践中，只有当 UI 需要实时展示参数时你才手工累积；否则你只要收集完整的 arguments 字符串、解析一次，然后完全按上一个 notebook 的方式继续工具循环（执行、追加 `role: "tool"` 结果、开一个新流）。

## 一个真实后端需要什么

在生产里，你很少是往终端里打印。你坐在模型和浏览器之间。有三件事 notebook 展示不了，但你应该在设计时考虑：

**1. 以 SSE 转发。** 不要在服务端把整个响应缓存起来；把 delta 重新作为你自己的 server-sent events（SSE）发出。示意如下（FastAPI 风格伪代码；别在 Colab 里运行这种单元格形态）：

```python
@app.post('/chat')
async def chat(req: ChatRequest):
    def gen():
        parts, usage = [], None
        stream = client.chat.completions.create(
            model=MODEL, max_tokens=1000, stream=True, messages=req.messages,
        )
        try:
            for chunk in stream:
                if chunk.choices and chunk.choices[0].delta.content:
                    text = chunk.choices[0].delta.content
                    parts.append(text)
                    yield f'data: {json.dumps({"delta": text})}\n\n'
                if getattr(chunk, 'x_groq', None) and chunk.x_groq.usage:
                    usage = chunk.x_groq.usage
        finally:
            # 即使客户端在流中途断开也会执行：
            record_usage(req.user_id, usage)   # billing truth
        yield 'data: [DONE]\n\n'
    return StreamingResponse(gen(), media_type='text/event-stream')
```

**2. 客户端断开。** 用户经常在答案生成到一半时关掉标签页。你的生成器被取消，但模型继续生成，你继续付费。要有意识地决定：断开时中止上游请求（省 token，丢掉答案），还是让它跑完并持久化结果（花 token，支持「恢复」）。无论哪种，`finally` 块都仍然必须把 usage 抓下来。

**3. 用于计费的 usage。** usage chunk 是*最后*才到的。如果你只转发文本 delta 而丢掉尾部那个 chunk，你就把账单扔了。永远在服务端读取 `x_groq.usage` chunk，并在请求处理器退出前记录下来。


## 练习

1. 扩展 TTFT 单元格，同时记录*token 间间隔*（相邻 delta 之间的时间）并打印 p50/p95。这个数字决定了流式输出是顺滑还是卡顿。
2. 写一个 `stream_to_list(prompt)`，返回 `(chunks, final_usage)`：按顺序返回每个文本 delta，外加最终的 usage 对象。验证拼接后的 chunks 等于完整组装出来的文本。
3. 把这个 notebook 和上一个结合起来：写一个 `run_agent_streaming` 循环，每回合都流式接收、实时打印文本 delta，然后从片段里重新组装 tool call，一直继续到 `finish_reason == 'stop'`。
4. 模拟客户端断开：在第 10 个文本 delta 之后跳出 chunk 循环。确认你丢掉了 usage chunk（它在最后），然后重构代码，让 `finally` 块仍然记录你到目前为止流式发出的内容。
